# Smart MCQ Solver (Gemma 4 Fine-tuned)

This notebook contains the codes for fine-tunning Gemma 4.

## Build By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Installation

In [1]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!uv pip install unsloth unsloth-zoo triton transformers peft bitsandbytes trl datasets

Using Python 3.12.13 environment at: /usr
Checked 8 packages in 163ms


## Imports

In [3]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import math
import re
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import wandb
from datasets import Dataset
from kaggle_secrets import UserSecretsClient  # type: ignore
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from transformers import TrainingArguments
from trl import SFTTrainer

In [5]:
warnings.filterwarnings("ignore")

## Configurations

In [ ]:
user_secrets = UserSecretsClient()

In [ ]:
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

In [6]:
# Dataset Path
DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

# Output paths
TRAIN_OUT_DIR = Path("/kaggle/working/gemma4-mcq-lora/")
TRAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUT_DIR = Path("/kaggle/working/gemma4-lora/")
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1")
MAX_SEQ_LEN = 512
LOAD_IN_4BIT = True

# Lora
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Training
VAL_FRAC = 0.2
EPOCHS = 10
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
WARMUP_RATIO = 0.05
LR_SCHEDULER = "cosine"

# Self RAG
TOP_K = 3

# Seed
SEED = 42

# Valid Choices
CHOICES = ["A", "B", "C", "D", "E"]

## Wandb Config

In [44]:
wandb.init(
    entity=user_secrets.get_secret("WANDB_ENTITY"),
    project=user_secrets.get_secret("WANDB_PROJECT"),
    notes="Gemma 4 E4B IT Finetune",
    config={
        # Paths
        "model_path": str(MODEL_PATH),
        "train_csv": str(TRAIN_CSV),
        "test_csv": str(TEST_CSV),
        # Model
        "max_seq_len": MAX_SEQ_LEN,
        "load_in_4bit": LOAD_IN_4BIT,
        # LoRA
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        # Training
        "val_frac": VAL_FRAC,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "effective_batch": BATCH_SIZE * GRAD_ACCUM,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler": LR_SCHEDULER,
        # Self-RAG
        "top_k_rag": TOP_K,
        # Misc
        "seed": SEED,
        "choices": CHOICES,
    },
)

NameError: name 'user_secrets' is not defined

## Helper Functions

In [7]:
def map_at_3(preds: List[List[str]], labels: List[str]) -> float:
    score = 0.0
    for pred, label in zip(preds, labels):
        for k, p in enumerate(pred[:3], 1):
            if p == label:
                score += 1.0 / k
                break

    return score / max(1, len(labels))

In [8]:
def set_seed(s: int):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

In [9]:
def build_user_message(row: Dict, context: str = "") -> str:
    """
    Builds the user-turn message.

    With context (self RAG):
      Context:
      Q: <similar question>
      Answer: A) <text>

      Q: ...
      Answer: ...

      Question: <current question>
      A) ...   B) ...   C) ...   D) ...   E) ...
      The correct answer is:

    Without context (zero-shot):
      Question: <current question>
      ...
    """
    context_block = f"Context:\n{context.strip()}\n\n" if context.strip() else ""
    return (
        f"{context_block}"
        f"Question: {row['prompt']}\n\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}\n\n"
        f"The correct answer is:"
    )

In [10]:
def format_for_sft(row: Dict, tokenizer, context: str = "") -> Dict:
    context = str(row.get("_context", ""))
    messages = [
        {"role": "user", "content": build_user_message(row, context)},
        {"role": "assistant", "content": str(row["answer"]).strip().upper()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

In [11]:
def clean(text: str) -> str:
    text = str(text).lower()  # type: ignore
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [12]:
def extract_core_question(prompt: str) -> str:
    """Extract core question by removing template prefixes and suffixes."""
    if ":" in prompt:
        q = prompt.split(":", 1)[1].strip()
    else:
        q = prompt.strip()

    if "?" in q:
        q = q.rsplit("?", 1)[0].strip() + "?"
    return clean(q)

In [13]:
@torch.inference_mode()
def score_and_rank(
    model,
    tokenizer,
    row: Dict,
    choice_token_ids: Dict[str, int],
    choices: List[str],
) -> List[str]:
    """
    Single forward pass - log-prob ranking of all 5 choices.
    """
    context = str(row.get("_context", ""))
    messages = [{"role": "user", "content": build_user_message(row, context)}]
    prefix_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # appends <start_of_turn>model\n
    )
    # print(prefix_str)
    input_ids = tokenizer(
        text=prefix_str,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(model.device)

    # Forward pass
    last_logits = model(input_ids).logits[0, -1, :]  # [vocab_size]
    log_probs = torch.log_softmax(last_logits, dim=-1)

    scores = {ch: log_probs[tid].item() for ch, tid in choice_token_ids.items()}
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:3]

In [14]:
def get_choice_token_ids(tokenizer, choices: List[str]) -> Dict[str, int]:
    """
    Resolves each answer label to its single vocabulary token ID.
    """
    choice_ids = {}
    for ch in choices:
        plain = tokenizer.tokenizer.encode(ch, add_special_tokens=False)
        spaced = tokenizer.tokenizer.encode(" " + ch, add_special_tokens=False)
        if len(plain) == 1:
            choice_ids[ch] = plain[0]
        elif len(spaced) == 1:
            choice_ids[ch] = spaced[0]
        else:
            choice_ids[ch] = plain[0]
            print(f"Warning: '{ch}' is multi-token ({plain}). Using first token.")

    decoded = {k: (v, tokenizer.tokenizer.decode([v])) for k, v in choice_ids.items()}
    print(f"Choice token IDs: {decoded}")
    return choice_ids

In [ ]:
def evaluate(
    model,
    tokenizer,
    val_df: pd.DataFrame,
    label: str = "Evaluation",
) -> float:
    """
    Runs inference on the val set (using pre-computed contexts
    from val_df["_context"]) and reports MAP@3 + accuracy breakdown.
    """
    FastLanguageModel.for_inference(model)
    model.eval()

    choice_ids = get_choice_token_ids(tokenizer, CHOICES)
    predictions, labels = [], []

    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=label):
        top3 = score_and_rank(model, tokenizer, row.to_dict(), choice_ids, CHOICES)
        predictions.append(top3)
        labels.append(row["answer"])

    map3 = float(np.mean([map_at_3(p, lbl) for p, lbl in zip(predictions, labels)]))

    n = len(labels)
    acc = {k: sum(a in p[:k] for a, p in zip(labels, predictions)) for k in [1, 2, 3]}

    print(f"\n{'-' * 45}")
    print(f"  {label}")
    print(f"{'-' * 45}")
    print(f"  MAP@3       :  {map3:.4f}")
    print(f"  Accuracy@1  :  {acc[1] / n:.4f}  ({acc[1]}/{n})")
    print(f"  Accuracy@2  :  {acc[2] / n:.4f}  ({acc[2]}/{n})")
    print(f"  Accuracy@3  :  {acc[3] / n:.4f}  ({acc[3]}/{n})")
    print(f"{'-' * 45}\n")
    return map3

## Retriever

In [16]:
class TFIDFRetriever:
    """
    TF-IDF retriever over the training corpus that provides few-shot
    context for each question.
    """

    def __init__(self, df: pd.DataFrame | pd.Series, top_k: int = 3) -> None:
        self.top_k = top_k
        self.corpus = df.reset_index(drop=True).copy()
        self.corpus["_core"] = self.corpus["prompt"].apply(extract_core_question)

        self.vectorizer = TfidfVectorizer(
            max_features=50_000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b\w\w+\b",
        )
        self.corpus_matrix = self.vectorizer.fit_transform(self.corpus["prompt"])
        print(
            f"TF-IDF index: {self.corpus_matrix.shape[0]} documents, "
            f"{self.corpus_matrix.shape[1]} features"
        )

    def retrieve(self, query: str) -> str:
        """
        Returns a formatted context string for one query prompt.

        Returned format (3 retrieved pairs, separated by blank lines):
          Q: <similar question>
          Answer: <label>) <full answer text>

          Q: ...
          Answer: ...
        """
        core_q = extract_core_question(query)
        qv = self.vectorizer.transform([core_q])
        sims = cosine_similarity(qv, self.corpus_matrix).flatten()

        order = sims.argsort()[::-1]
        ctx_parts = []
        for idx in order:
            row = self.corpus.iloc[idx]
            if row["_core"] == core_q:
                continue

            label = str(row["answer"]).strip().upper()
            answer_text = str(row[label]).strip()
            ctx_parts.append(f"Q: {row['prompt']}\nAnswer: {label}) {answer_text}")
            if len(ctx_parts) >= self.top_k:
                break

        return "\n\n".join(ctx_parts)

    def precompute_contexts(self, df: pd.DataFrame, desc: str) -> List[str]:
        return [
            self.retrieve(row["prompt"])
            for _, row in tqdm(df.iterrows(), total=len(df), desc=desc)
        ]

## Pre-processing

In [17]:
df = pd.read_csv(TRAIN_CSV)
df["core_question"] = df["prompt"].apply(extract_core_question)
unique_cores = df["core_question"].unique()

In [18]:
rng = np.random.default_rng(SEED)
rng.shuffle(unique_cores)

In [19]:
n_val_cores = max(1, int(len(unique_cores) * VAL_FRAC))
val_cores = set(unique_cores[:n_val_cores])

In [20]:
df_val = df[df["core_question"].isin(val_cores)].reset_index(drop=True)
df_train = df[~df["core_question"].isin(val_cores)].reset_index(drop=True)

print(f"Data Split: train={len(df_train)} samples, validation={len(df_val)} samples")

Data Split: train=1623 samples, validation=377 samples


In [21]:
print("Fitting TF-IDF retriever on train split...")
retriever = TFIDFRetriever(df_train, top_k=TOP_K)

Fitting TF-IDF retriever on train split...
TF-IDF index: 1623 documents, 2782 features


In [ ]:
print("Pre-computing RAG contexts for data")
df_train["_context"] = retriever.precompute_contexts(
    df_train, "Retrieving contexts for train split"
)  # noqa: E501
df_val["_context"] = retriever.precompute_contexts(
    df_val, "Retrieving contexts for val split"
)  # noqa: E501

Pre-computing RAG contexts for data


Retrieving contexts for val split: 100%|██████████| 377/377 [00:00<00:00, 483.83it/s]


In [23]:
df_train.sample(5)

,id,prompt,A,B,C,D,E,answer,core_question,_context
1347,1654,Identify the correct statement: What is the de...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,B,what is the definition of an improper rotation,Q: Which of the following is correct? What is ...
1276,1571,Choose the correct answer: What is the estimat...,"Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",B,what is the estimated redshift of ceers 93316 ...,Q: Which of the following is correct? What is ...
1449,1783,Choose the correct answer: What are the two ma...,The universe began with a small preference for...,The universe began with a small preference for...,The universe began with equal amounts of matte...,The universe began with a small preference for...,The universe began with equal amounts of matte...,D,what are the two main interpretations for the ...,Q: What are the two main factors that cause re...
1558,1921,Choose the correct answer: What is the gravito...,The gravitomagnetic interaction is a force tha...,The gravitomagnetic interaction is a force tha...,The gravitomagnetic interaction is a new force...,The gravitomagnetic interaction is a force tha...,The gravitomagnetic interaction is a force tha...,C,what is the gravitomagnetic interaction,Q: Which of the following is correct? What is ...
996,1234,Identify the correct statement: What is the ap...,Memristor has applications in the production o...,Memristor has applications in the production o...,Memristor has applications in the production o...,Memristor has applications in programmable log...,Memristor has applications in optical fiber co...,D,what is the application of memristor,Q: Which of the following is correct? What is ...


## Model Loading

In [24]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MODEL_PATH),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    device_map={"": 0},
)

==((====))==  Unsloth 2026.6.1: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [25]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


In [26]:
model.config.use_cache = False

In [27]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

In [28]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  ({100 * trainable / total:.2f}%)\n")

Total params    : 5,989,059,104
Trainable params: 6,823,936  (0.11%)



## Zero-shot Baseline

In [29]:
# evaluate(model, tokenizer, df_val, label="Zero-shot + context")

## Fine-tuning

In [30]:
FastLanguageModel.for_training(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0-4): 5 x Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=8, out_features=512, 

In [31]:
formatted_rows = [
    format_for_sft(row.to_dict(), tokenizer) for _, row in df_train.iterrows()
]
train_dataset = Dataset.from_list(formatted_rows)

In [32]:
total_steps = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

In [ ]:
training_args = TrainingArguments(
    output_dir=TRAIN_OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=warmup_steps,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    report_to="wandb",
    logging_steps=20,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
)

In [34]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1623 [00:00<?, ? examples/s]

In [35]:
print(
    f"Training:  {len(train_dataset)} examples  |  "
    f"{EPOCHS} epochs  |  "
    f"effective batch = {BATCH_SIZE * GRAD_ACCUM}\n"
)
trainer.train()
print("Training complete.\n")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


Training:  1623 examples  |  10 epochs  |  effective batch = 16



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,623 | Num Epochs = 10 | Total steps = 510
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 6,823,936 of 8,002,980,384 (0.09% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
map3 = evaluate(model, tokenizer, df_val, label="Fine-tuned + context")
map3

In [ ]:
wandb.log({"eval/map@3": map3})
wandb.finish()

## Save Model

In [ ]:
model.save_pretrained(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)

In [ ]:
print("Fitting TF-IDF retriever on full dataset...")
retriever = TFIDFRetriever(df, top_k=TOP_K)
retriever.save(MODEL_OUT_DIR / "retriever.pkl")